# Import Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

 نعمل Request لأول صفحة

In [2]:
url = "https://books.toscrape.com/"

response = requests.get(url)

print(response.status_code)

200


**نتأكد إننا جبنا الـ HTML**

In [3]:
print(response.text[:1000])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:29" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        
            <link rel="shortcut icon" href="static/oscar/favicon.

**تحويل الـ HTML لـ Soup**

In [4]:
soup = BeautifulSoup(response.text, "html.parser")

**نحدد كل الكتب الموجودة في الصفحة**

In [5]:
books = soup.find_all("article", class_="product_pod")

print(len(books))

20


**نبص على أول كتاب ونفهم الـ HTML بتاعه**

In [6]:
print(books[0])

<article class="product_pod">
<div class="image_container">
<a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">Â£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>


# Step 7 — Extract Book Title

In [7]:
title = books[0].find("h3").find("a")["title"]

print(title)

A Light in the Attic


**Step 8 — Extract Price**

In [8]:
price = books[0].find("p", class_="price_color").text

print(price)

Â£51.77


**Step 9 — Extract Availability**

In [9]:
availability = books[0].find("p", class_="instock availability").text.strip()

print(availability)

In stock


**Step 10 — نجمع بيانات أول كتاب في Dictionary**

In [10]:
book_data = {
    "Name": title,
    "Price": price,
    "Availability": availability
}

print(book_data)

{'Name': 'A Light in the Attic', 'Price': 'Â£51.77', 'Availability': 'In stock'}


لسه ناقصنا الـ Category.

الـ Category مش موجودة مباشرة في كارت الكتاب في الصفحة الرئيسية؛ هنحتاج ندخل على صفحة الكتاب نفسه عشان نجيبها من هناك.

هنجيب الـ URL الخاص بأول كتاب، ونعمل عليه Request تاني، وبعدها نستخرج الـ Category.

In [11]:
book_link = books[0].find("h3").find("a")["href"]

print(book_link)

catalogue/a-light-in-the-attic_1000/index.html


**Step 11 — نجيب اللينك كامل باستخدام urljoin**

In [12]:
from urllib.parse import urljoin

book_url = urljoin(url, book_link)

print(book_url)

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


# Step 12 — نفتح صفحة الكتاب نفسها

In [13]:
book_response = requests.get(book_url)

print(book_response.status_code)

200


In [14]:
book_soup = BeautifulSoup(book_response.text, "html.parser")

**# Step 13 — نطلع الـ Category**

In [15]:
category = book_soup.find("ul", class_="breadcrumb").find_all("li")[-2].text.strip()

print(category)

Poetry


# Step 14 — نجمع الأربع بيانات في Dictionary واحد

In [16]:
book_data = {
    "Category": category,
    "Name": title,
    "Price": price,
    "Availability": availability
}

print(book_data)

{'Category': 'Poetry', 'Name': 'A Light in the Attic', 'Price': 'Â£51.77', 'Availability': 'In stock'}


# Step 15 — نفهم إزاي هنكرر العملية

# Step 16 — نعمل Loop على أول صفحة (20 كتاب)

In [17]:
all_books = []

for book in books:
    title = book.find("h3").find("a").get("title", "Unknown")

    price = book.find("p", class_="price_color").text.strip()

    availability = book.find("p", class_="instock availability").text.strip()

    book_link = book.find("h3").find("a").get("href", "Unknown")

    book_url = urljoin(url, book_link)

    book_response = requests.get(book_url)

    book_soup = BeautifulSoup(book_response.text, "html.parser")

    category = book_soup.find("ul", class_="breadcrumb").find_all("li")[-2].text.strip()

    book_data = {
        "Category": category,
        "Name": title,
        "Price": price,
        "Availability": availability
    }

    all_books.append(book_data)

# Step 17 — نتأكد إن الـ Loop اشتغلت

In [18]:
print(len(all_books))

20


In [19]:
all_books[:2]

[{'Category': 'Poetry',
  'Name': 'A Light in the Attic',
  'Price': 'Â£51.77',
  'Availability': 'In stock'},
 {'Category': 'Historical Fiction',
  'Name': 'Tipping the Velvet',
  'Price': 'Â£53.74',
  'Availability': 'In stock'}]

**Step 18 — نفهم الـ Pagination**

**Step 19 — نجرب نفتح الصفحة التانية**

In [22]:
page_url = "https://books.toscrape.com/catalogue/page-2.html"

page_response = requests.get(page_url)

print(page_response.status_code)


200


In [23]:
page_soup = BeautifulSoup(page_response.text, "html.parser")

page_books = page_soup.find_all("article", class_="product_pod")

print(len(page_books))

20


### Scraping All 50 Pages

In this section, we loop through all 50 pages of the website and extract the required information from each book, including its category, name, price, and availability. All extracted data is stored in a single list called `all_books`.

In [24]:
all_books = []

for page in range(1, 51):

    if page == 1:
        page_url = url
    else:
        page_url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    page_response = requests.get(page_url)

    page_soup = BeautifulSoup(page_response.text, "html.parser")

    books = page_soup.find_all("article", class_="product_pod")

    for book in books:

        title = book.find("h3").find("a").get("title", "Unknown")

        price = book.find("p", class_="price_color").text.strip()

        availability = book.find("p", class_="instock availability").text.strip()

        book_link = book.find("h3").find("a").get("href", "Unknown")

        book_url = urljoin(page_url, book_link)

        book_response = requests.get(book_url)

        book_soup = BeautifulSoup(book_response.text, "html.parser")

        category_tag = book_soup.find("ul", class_="breadcrumb")

        if category_tag:
            category_items = category_tag.find_all("li")

            if len(category_items) >= 2:
                category = category_items[-2].text.strip()
            else:
                category = "Unknown"
        else:
            category = "Unknown"

        book_data = {
            "Category": category,
            "Name": title,
            "Price": price,
            "Availability": availability
        }

        all_books.append(book_data)

    print(f"Page {page} completed - Total books: {len(all_books)}")

Page 1 completed - Total books: 20
Page 2 completed - Total books: 40
Page 3 completed - Total books: 60
Page 4 completed - Total books: 80
Page 5 completed - Total books: 100
Page 6 completed - Total books: 120
Page 7 completed - Total books: 140
Page 8 completed - Total books: 160
Page 9 completed - Total books: 180
Page 10 completed - Total books: 200
Page 11 completed - Total books: 220
Page 12 completed - Total books: 240
Page 13 completed - Total books: 260
Page 14 completed - Total books: 280
Page 15 completed - Total books: 300
Page 16 completed - Total books: 320
Page 17 completed - Total books: 340
Page 18 completed - Total books: 360
Page 19 completed - Total books: 380
Page 20 completed - Total books: 400
Page 21 completed - Total books: 420
Page 22 completed - Total books: 440
Page 23 completed - Total books: 460
Page 24 completed - Total books: 480
Page 25 completed - Total books: 500
Page 26 completed - Total books: 520
Page 27 completed - Total books: 540
Page 28 comple

### Checking the Scraped Data

In this section, we check the total number of scraped books and display a sample of the extracted data to ensure that the scraping process was successful.

In [25]:
print("Total Books:", len(all_books))

Total Books: 1000


In [26]:
all_books[:5]

[{'Category': 'Poetry',
  'Name': 'A Light in the Attic',
  'Price': 'Â£51.77',
  'Availability': 'In stock'},
 {'Category': 'Historical Fiction',
  'Name': 'Tipping the Velvet',
  'Price': 'Â£53.74',
  'Availability': 'In stock'},
 {'Category': 'Fiction',
  'Name': 'Soumission',
  'Price': 'Â£50.10',
  'Availability': 'In stock'},
 {'Category': 'Mystery',
  'Name': 'Sharp Objects',
  'Price': 'Â£47.82',
  'Availability': 'In stock'},
 {'Category': 'History',
  'Name': 'Sapiens: A Brief History of Humankind',
  'Price': 'Â£54.23',
  'Availability': 'In stock'}]

In [27]:
all_books[-5:]

[{'Category': 'Classics',
  'Name': "Alice in Wonderland (Alice's Adventures in Wonderland #1)",
  'Price': 'Â£55.53',
  'Availability': 'In stock'},
 {'Category': 'Sequential Art',
  'Name': 'Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)',
  'Price': 'Â£57.06',
  'Availability': 'In stock'},
 {'Category': 'Historical Fiction',
  'Name': "A Spy's Devotion (The Regency Spies of London #1)",
  'Price': 'Â£16.97',
  'Availability': 'In stock'},
 {'Category': 'Mystery',
  'Name': "1st to Die (Women's Murder Club #1)",
  'Price': 'Â£53.98',
  'Availability': 'In stock'},
 {'Category': 'Travel',
  'Name': '1,000 Places to See Before You Die',
  'Price': 'Â£26.08',
  'Availability': 'In stock'}]

### Converting Data into a DataFrame

In this section, we convert the scraped data into a Pandas DataFrame to organize the information into a structured table.

In [28]:
df = pd.DataFrame(all_books)

df.head()

,Category,Name,Price,Availability
0,Poetry,A Light in the Attic,Â£51.77,In stock
1,Historical Fiction,Tipping the Velvet,Â£53.74,In stock
2,Fiction,Soumission,Â£50.10,In stock
3,Mystery,Sharp Objects,Â£47.82,In stock
4,History,Sapiens: A Brief History of Humankind,Â£54.23,In stock


In [29]:
df.shape

(1000, 4)

### Exporting Data to CSV

In this section, we export the final DataFrame into a CSV file to save the scraped book information.

In [30]:
df.to_csv("books_data.csv", index=False, encoding="utf-8-sig")

In [31]:
from google.colab import files

files.download("books_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>